# Extracción de datos del Mineduc


In [1]:
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service

### Importar datos

In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service

from webdriver_manager.chrome import ChromeDriverManager

# Configuración de Chrome
options = Options()

# Crear el navegador
driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

# Abrir la página
driver.get("https://www.mineduc.gob.gt/BUSCAESTABLECIMIENTO_GE/")

# Maximizar la ventana
driver.maximize_window()

In [3]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

wait = WebDriverWait(driver, 20)

departamento = wait.until(
    EC.presence_of_element_located(
        (By.NAME, "_ctl0:ContentPlaceHolder1:cmbDepartamento")
    )
)

print("Página cargada correctamente.")

Página cargada correctamente.


In [4]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC

wait = WebDriverWait(driver, 20)

# Esperar a que aparezca el combo de departamentos
combo_departamento = wait.until(
    EC.presence_of_element_located(
        (By.NAME, "_ctl0:ContentPlaceHolder1:cmbDepartamento")
    )
)

# Seleccionar GUATEMALA
Select(combo_departamento).select_by_visible_text("GUATEMALA")

print("Departamento seleccionado.")

Departamento seleccionado.


In [5]:
import time

time.sleep(2)

# Seleccionar TODOS en municipio
combo_municipio = Select(driver.find_element(
    By.NAME,
    "_ctl0:ContentPlaceHolder1:cmbMunicipio"
))

combo_municipio.select_by_visible_text("TODOS")

print("Municipio seleccionado.")

Municipio seleccionado.


In [6]:
Select(driver.find_element(
    By.NAME,
    "_ctl0:ContentPlaceHolder1:cmbNivel"
)).select_by_visible_text("TODOS")

Select(driver.find_element(
    By.NAME,
    "_ctl0:ContentPlaceHolder1:cmbSector"
)).select_by_visible_text("TODOS")

Select(driver.find_element(
    By.NAME,
    "_ctl0:ContentPlaceHolder1:ddlplan"
)).select_by_visible_text("TODOS")

Select(driver.find_element(
    By.NAME,
    "_ctl0:ContentPlaceHolder1:ddlModalidad"
)).select_by_visible_text("TODOS")

print("Filtros seleccionados.")

Filtros seleccionados.


In [7]:
driver.find_element(
    By.NAME,
    "_ctl0:ContentPlaceHolder1:IbtnConsultar"
).click()

print("Consulta enviada.")

Consulta enviada.


## Leer tabla de centros educativos para Guatemala con Selenium

In [8]:
tabla = driver.find_element(
    By.ID,
    "_ctl0_ContentPlaceHolder1_dgResultado"
)

print(tabla.get_attribute("outerHTML")[:500])

<table cellspacing="0" rules="all" bordercolor="SteelBlue" border="1" id="_ctl0_ContentPlaceHolder1_dgResultado" style="border-color:SteelBlue;font-family:Arial;font-size:Smaller;height:104px;width:100%;border-collapse:collapse;">
	<tbody><tr align="center" valign="middle" style="background-color:LightSteelBlue;">
		<td>&nbsp;</td><td>CODIGO</td><td>DISTRITO</td><td>DEPARTAMENTO</td><td>MUNICIPIO</td><td>ESTABLECIMIENTO</td><td>DIRECCION</td><td>TELEFONO</td><td>SUPERVISOR</td><td>DIRECTOR</td><


In [9]:
from io import StringIO
import pandas as pd

html = tabla.get_attribute("outerHTML")

df = pd.read_html(StringIO(html))[0]

df.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
0,NaN,CODIGO,DISTRITO,DEPARTAMENTO,MUNICIPIO,ESTABLECIMIENTO,DIRECCION,TELEFONO,SUPERVISOR,DIRECTOR,NIVEL,SECTOR,AREA,STATUS,MODALIDAD,JORNADA,PLAN,DEPARTAMENTAL
1,NaN,01-01-0001-42,01-110,GUATEMALA,GUATEMALA,LICEO CIENTIFICO CUMORAH,LOTE 572 MANZANA 5 COLONIA MAYA ZONA 18,22603791,ZUHILEM YESENIA ESTRADA ORTIZ,BERLY MARISOL MORALES OCHOA,PARVULOS,PRIVADO,URBANA,CERRADA TEMPORALMENTE,MONOLINGUE,MATUTINA,DIARIO(REGULAR),GUATEMALA NORTE
2,NaN,01-01-0002-43,01-110,GUATEMALA,GUATEMALA,LICEO CIENTIFICO CUMORAH,LOTE 572 MANZANA 5 COLONIA MAYA ZONA 18,22603601,ZUHILEM YESENIA ESTRADA ORTIZ,BERLY MARISOL MORALES OCHOA,PRIMARIA,PRIVADO,URBANA,CERRADA TEMPORALMENTE,MONOLINGUE,MATUTINA,DIARIO(REGULAR),GUATEMALA NORTE
3,NaN,01-01-0003-45,01-110,GUATEMALA,GUATEMALA,LICEO CIENTIFICO CUMORAH,LOTE 572 MANZANA 5 COLONIA MAYA ZONA 18,22604386,ZUHILEM YESENIA ESTRADA ORTIZ,BERLY MARISOL MORALES OCHOA,BASICO,PRIVADO,URBANA,CERRADA TEMPORALMENTE,MONOLINGUE,MATUTINA,DIARIO(REGULAR),GUATEMALA NORTE
4,NaN,01-01-0004-42,01-504,GUATEMALA,GUATEMALA,COLEGIO HAN AL AMERICANO,9A. CALLE 28-74 ZONA 14,23661600,DAVID SOTOJ SANCHEZ,ANGELICA MARIA REYES MONTERROSO,PARVULOS,PRIVADO,URBANA,CERRADA DEFINITIVAMENTE,MONOLINGUE,MATUTINA,DIARIO(REGULAR),GUATEMALA NORTE


## Limpieza de datos 

In [10]:
# La primera fila contiene los encabezados
df.columns = df.iloc[0]

# Eliminar esa primera fila
df = df.iloc[1:].reset_index(drop=True)

df.head()

# Eliminar la primera columna (la del botón "Seleccionar Establecimiento")
df = df.drop(df.columns[0], axis=1)

df.head()


,CODIGO,DISTRITO,DEPARTAMENTO,MUNICIPIO,ESTABLECIMIENTO,DIRECCION,TELEFONO,SUPERVISOR,DIRECTOR,NIVEL,SECTOR,AREA,STATUS,MODALIDAD,JORNADA,PLAN,DEPARTAMENTAL
0,01-01-0001-42,01-110,GUATEMALA,GUATEMALA,LICEO CIENTIFICO CUMORAH,LOTE 572 MANZANA 5 COLONIA MAYA ZONA 18,22603791,ZUHILEM YESENIA ESTRADA ORTIZ,BERLY MARISOL MORALES OCHOA,PARVULOS,PRIVADO,URBANA,CERRADA TEMPORALMENTE,MONOLINGUE,MATUTINA,DIARIO(REGULAR),GUATEMALA NORTE
1,01-01-0002-43,01-110,GUATEMALA,GUATEMALA,LICEO CIENTIFICO CUMORAH,LOTE 572 MANZANA 5 COLONIA MAYA ZONA 18,22603601,ZUHILEM YESENIA ESTRADA ORTIZ,BERLY MARISOL MORALES OCHOA,PRIMARIA,PRIVADO,URBANA,CERRADA TEMPORALMENTE,MONOLINGUE,MATUTINA,DIARIO(REGULAR),GUATEMALA NORTE
2,01-01-0003-45,01-110,GUATEMALA,GUATEMALA,LICEO CIENTIFICO CUMORAH,LOTE 572 MANZANA 5 COLONIA MAYA ZONA 18,22604386,ZUHILEM YESENIA ESTRADA ORTIZ,BERLY MARISOL MORALES OCHOA,BASICO,PRIVADO,URBANA,CERRADA TEMPORALMENTE,MONOLINGUE,MATUTINA,DIARIO(REGULAR),GUATEMALA NORTE
3,01-01-0004-42,01-504,GUATEMALA,GUATEMALA,COLEGIO HAN AL AMERICANO,9A. CALLE 28-74 ZONA 14,23661600,DAVID SOTOJ SANCHEZ,ANGELICA MARIA REYES MONTERROSO,PARVULOS,PRIVADO,URBANA,CERRADA DEFINITIVAMENTE,MONOLINGUE,MATUTINA,DIARIO(REGULAR),GUATEMALA NORTE
4,01-01-0005-43,01-504,GUATEMALA,GUATEMALA,COLEGIO HAN AL AMERICANO,9A. CALLE 28-74 ZONA 14,23661600,DAVID SOTOJ SANCHEZ,LOURDES YAJAIRA GARCIA ROSALES DE YANES,PRIMARIA,PRIVADO,URBANA,CERRADA DEFINITIVAMENTE,MONOLINGUE,MATUTINA,DIARIO(REGULAR),GUATEMALA NORTE


## Convertir datos a archivo CSV

In [11]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from io import StringIO
import pandas as pd
import time

def consultar(driver, departamento, nivel):

    wait = WebDriverWait(driver, 20)

    # Volver a la página principal
    driver.get("https://www.mineduc.gob.gt/BUSCAESTABLECIMIENTO_GE/")

    # Esperar que cargue el formulario
    wait.until(
        EC.presence_of_element_located(
            (By.NAME, "_ctl0:ContentPlaceHolder1:cmbDepartamento")
        )
    )

    # Departamento
    Select(driver.find_element(
        By.NAME,
        "_ctl0:ContentPlaceHolder1:cmbDepartamento"
    )).select_by_visible_text(departamento)

    # Esperar que actualice el municipio
    time.sleep(2)

    # Municipio
    Select(driver.find_element(
        By.NAME,
        "_ctl0:ContentPlaceHolder1:cmbMunicipio"
    )).select_by_visible_text("TODOS")

    # Nivel
    Select(driver.find_element(
        By.NAME,
        "_ctl0:ContentPlaceHolder1:cmbNivel"
    )).select_by_visible_text(nivel)

    # Sector
    Select(driver.find_element(
        By.NAME,
        "_ctl0:ContentPlaceHolder1:cmbSector"
    )).select_by_visible_text("TODOS")

    # Plan
    Select(driver.find_element(
        By.NAME,
        "_ctl0:ContentPlaceHolder1:ddlplan"
    )).select_by_visible_text("TODOS")

    # Modalidad
    Select(driver.find_element(
        By.NAME,
        "_ctl0:ContentPlaceHolder1:ddlModalidad"
    )).select_by_visible_text("TODOS")

    # Buscar
    driver.find_element(
        By.NAME,
        "_ctl0:ContentPlaceHolder1:IbtnConsultar"
    ).click()

    # Esperar la tabla
    wait.until(
        EC.presence_of_element_located(
            (By.ID, "_ctl0_ContentPlaceHolder1_dgResultado")
        )
    )

    # Extraer HTML
    tabla = driver.find_element(
        By.ID,
        "_ctl0_ContentPlaceHolder1_dgResultado"
    )

    html = tabla.get_attribute("outerHTML")

    # Convertir a DataFrame
    df = pd.read_html(StringIO(html))[0]

    # Limpiar
    df.columns = df.iloc[0]
    df = df.iloc[1:].reset_index(drop=True)
    df = df.drop(df.columns[0], axis=1)

    return df

In [12]:
departamentos = [
    "GUATEMALA",
    "EL PROGRESO",
    "SACATEPEQUEZ",
    "CHIMALTENANGO",
    "ESCUINTLA",
    "SANTA ROSA",
    "SOLOLA",
    "TOTONICAPAN",
    "QUETZALTENANGO",
    "SUCHITEPEQUEZ",
    "RETALHULEU",
    "SAN MARCOS",
    "HUEHUETENANGO",
    "QUICHE",
    "BAJA VERAPAZ",
    "ALTA VERAPAZ",
    "PETEN",
    "IZABAL",
    "ZACAPA",
    "CHIQUIMULA",
    "JALAPA",
    "JUTIAPA"
]
niveles = [
    "DIVERSIFICADO"
]

todos = []

for depto in departamentos:

    for nivel in niveles:

        print(f"Consultando {depto} - {nivel}")

        df = consultar(driver, depto, nivel)

        todos.append(df)

Consultando GUATEMALA - DIVERSIFICADO
Consultando EL PROGRESO - DIVERSIFICADO
Consultando SACATEPEQUEZ - DIVERSIFICADO
Consultando CHIMALTENANGO - DIVERSIFICADO
Consultando ESCUINTLA - DIVERSIFICADO
Consultando SANTA ROSA - DIVERSIFICADO
Consultando SOLOLA - DIVERSIFICADO
Consultando TOTONICAPAN - DIVERSIFICADO
Consultando QUETZALTENANGO - DIVERSIFICADO
Consultando SUCHITEPEQUEZ - DIVERSIFICADO
Consultando RETALHULEU - DIVERSIFICADO
Consultando SAN MARCOS - DIVERSIFICADO
Consultando HUEHUETENANGO - DIVERSIFICADO
Consultando QUICHE - DIVERSIFICADO
Consultando BAJA VERAPAZ - DIVERSIFICADO
Consultando ALTA VERAPAZ - DIVERSIFICADO
Consultando PETEN - DIVERSIFICADO
Consultando IZABAL - DIVERSIFICADO
Consultando ZACAPA - DIVERSIFICADO
Consultando CHIQUIMULA - DIVERSIFICADO
Consultando JALAPA - DIVERSIFICADO
Consultando JUTIAPA - DIVERSIFICADO


In [13]:
df_final = pd.concat(todos, ignore_index=True)

df_final.to_csv(
    "establecimientos.csv",
    index=False,
    encoding="utf-8-sig"
)